In [110]:
import jax
import jax.numpy as jnp
import equinox as eqx
import optimistix as optx

key = jax.random.PRNGKey(0)

key, subkey = jax.random.split(key)
x = jnp.linspace(-1, 1, 100)[:, None]
y = 2 * x**2 - x + 1 + 0.1 * jax.random.normal(subkey, (100, 1))

In [111]:

# 2. Define the Equinox model
class MyModel(eqx.Module):
    layers: list
    flag: bool

    def __init__(self, key, flag=False):
        self.flag = flag
        key1, key2, key3 = jax.random.split(key, 3)
        self.layers = [
            eqx.nn.Linear(1, 16, key=key1),
            eqx.nn.Lambda(jax.nn.relu),
            eqx.nn.Linear(16, 16, key=key2),
            eqx.nn.Lambda(jax.nn.relu),
            eqx.nn.Linear(16, 1, key=key3),
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Instantiate the model
key, subkey = jax.random.split(key)
model = MyModel(subkey)

In [117]:
params, static = eqx.partition(model, eqx.is_array)

In [113]:
def loss_fn(params, args):
    static, x_data, y_data = args
    model = eqx.combine(params, static)
    y_pred = jax.vmap(model)(x_data)
    return jnp.mean((y_pred - y_data) ** 2)

In [114]:
solver = optx.BFGS(rtol=1e-5, atol=1e-5)
args = (static, x, y)
sol = optx.minimise(loss_fn, solver, params, args=args)

In [115]:
optimised_params = sol.value
optimised_model = eqx.combine(optimised_params, static)

final_loss = loss_fn(optimised_params, args)
print(f"Final loss: {final_loss}")

Final loss: 0.007878043688833714
